# ch04 时间趋势分析

分析点击率随时间的变化趋势，检测新奇效应。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Config
from src.utils.data_loader import DataLoader
from src.utils.visualizer import Visualizer
from src.utils.output_manager import OutputManager

print("模块加载成功")

## Step 1: 加载数据

In [ ]:
config = Config()
loader = DataLoader(config)
visualizer = Visualizer(config)
output = OutputManager(config)

df = loader.load_processed("cleaned_data.csv")
print(f"数据形状: {df.shape}")
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")

## Step 2: 按天分析点击率趋势

In [ ]:
# 提取日期
df['date'] = df['timestamp'].dt.date

# 按日期和分组聚合
daily_stats = df.groupby(['date', 'group']).agg({
    'user_id': 'count',
    'click': ['sum', 'mean']
}).reset_index()
daily_stats.columns = ['date', 'group', 'users', 'clicks', 'ctr']

print("按天点击率统计:")
print(daily_stats.head(10))

output.save_table(daily_stats, "daily_ctr_trend", chapter_prefix="ch04")

## Step 3: 按小时分析点击率趋势

In [ ]:
# 提取小时
df['hour'] = df['timestamp'].dt.hour

# 按小时和分组聚合
hourly_stats = df.groupby(['hour', 'group']).agg({
    'user_id': 'count',
    'click': ['sum', 'mean']
}).reset_index()
hourly_stats.columns = ['hour', 'group', 'users', 'clicks', 'ctr']

print("按小时点击率统计:")
print(hourly_stats.head(10))

output.save_table(hourly_stats, "hourly_ctr_trend", chapter_prefix="ch04")

## Step 4: 绘制时间趋势图

In [ ]:
# 按天趋势图
fig, ax = plt.subplots(figsize=(12, 6))

for group, color in zip(['con', 'exp'], ['#DD8452', '#4C72B0']):
    data = daily_stats[daily_stats['group'] == group]
    ax.plot(data['date'], data['ctr'], marker='o', label=group.upper(), color=color)

ax.set_title('A/B测试点击率按天趋势', fontsize=14)
ax.set_xlabel('日期', fontsize=12)
ax.set_ylabel('点击率', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
visualizer.save_figure(fig, "daily_ctr_trend", chapter_prefix="ch04")
plt.show()

## Step 5: 新奇效应检测

In [ ]:
# 实验组数据
exp_daily = daily_stats[daily_stats['group'] == 'exp']['ctr'].values

# 分为前半期和后半期
mid_point = len(exp_daily) // 2
first_half = np.mean(exp_daily[:mid_point])
second_half = np.mean(exp_daily[mid_point:])

# 计算差异
diff_pct = (first_half - second_half) / first_half * 100

print(f"前半期平均 CTR: {first_half:.4f}")
print(f"后半期平均 CTR: {second_half:.4f}")
print(f"差异比例: {diff_pct:.2f}%")

if diff_pct > 10:
    print("结论: 可能存在新奇效应")
elif diff_pct < -10:
    print("结论: 可能存在学习效应")
else:
    print("结论: 趋势稳定，无明显新奇效应或学习效应")